####1. Set catalog and schema

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####2. Create the employees table

In [0]:
%sql
CREATE OR REPLACE TABLE employees (
  employee_id  INT     COMMENT 'Unique employee identifier',
  name         STRING  COMMENT 'Full name',
  department   STRING  COMMENT 'Department name',
  salary       DOUBLE  COMMENT 'Annual salary in USD',
  status       STRING  COMMENT 'active or terminated'
)
USING DELTA
COMMENT 'Employee records — used for time travel demo';

####3. Initial load: 8 employees

In [0]:
%sql
INSERT INTO employees VALUES
  (1,  'Alice Nguyen',    'Engineering', 95000.00, 'active'),
  (2,  'Bob Patel',       'Engineering', 88000.00, 'active'),
  (3,  'Carol Santos',    'Engineering', 92000.00, 'active'),
  (4,  'David Kim',       'Engineering', 78000.00, 'active'),
  (5,  'Eva Müller',      'Marketing',   72000.00, 'active'),
  (6,  'Frank Osei',      'Marketing',   68000.00, 'active'),
  (7,  'Grace Lin',       'Marketing',   74000.00, 'active'),
  (8,  'Hiro Yamamoto',   'Marketing',   69000.00, 'active');

####4. Engineering salary adjustment: 10% raise

In [0]:
%sql
UPDATE employees
SET    salary = ROUND(salary * 1.10, 2)
WHERE  department = 'Engineering';

####5. New hire

In [0]:
%sql
INSERT INTO employees VALUES
  (9, 'Ingrid Larsson', 'Engineering', 85000.00, 'active');

####6. Employee termination

In [0]:
%sql
UPDATE employees
SET    status = 'terminated'
WHERE  employee_id = 6;

####7. Confirm history with DESCRIBE HISTORY

In [0]:
%sql
DESCRIBE HISTORY employees;

### Time travel demo

In [0]:
%sql
select * from employees

####1. VERSION AS OF: before the raise

In [0]:
%sql
SELECT   employee_id, name, department, salary, status
FROM     employees VERSION AS OF 1
ORDER BY department, employee_id;

####2. VERSION AS OF: after the raise

In [0]:
%sql
SELECT   employee_id, name, department, salary, status
FROM     employees VERSION AS OF 2
ORDER BY department, employee_id;

####3. VERSION AS OF: current state for comparison

In [0]:
%sql
SELECT   employee_id, name, department, salary, status
FROM     employees
ORDER BY department, employee_id;

####4. TIMESTAMP AS OF

In [0]:
%sql
SELECT   employee_id, name, department, salary, status
FROM     employees TIMESTAMP AS OF '2026-09-21T13:59:58.000+00:00'
ORDER BY department, employee_id;

####5. PySpark time travel

In [0]:
df_v1 = spark.read \
    .format("delta") \
    .option("versionAsOf", 1) \
    .table("employees")

print(f"Version 1 row count: {df_v1.count()}")
display(df_v1.orderBy("department", "employee_id"))

####6. Audit scenario: who earned over $80K before the raise?

In [0]:
%sql
SELECT   employee_id,
         name,
         department,
         salary AS salary_before_raise,
         ROUND(salary * 1.10, 2) AS salary_after_raise
FROM     employees VERSION AS OF 1
WHERE    department = 'Engineering'
AND      salary > 80000
ORDER BY salary DESC;